#### Install required libraries

In [1]:
#!pip install seaborn

### Import libraries

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

In [3]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)

In [4]:
pwd

'D:\\Learn\\deep learning\\realtime-football-player-tracking'

#### load dataset

In [5]:
worldcupstat = pd.read_csv('dataset/worldcupstat.csv')

#### Preview the datasets

In [6]:
worldcupstat.head()

,match_id,date,tournament,stage,team,opponent,team_goals,opponent_goals,result,shots,...,opponent_shots_on_target,opponent_possession_pct,opponent_passes,opponent_pass_accuracy_pct,opponent_fouls,opponent_yellow_cards,opponent_red_cards,opponent_offsides,opponent_corners,stats_period
0,WC2026_MEX_RSA_20260611,2026-06-11,FIFA World Cup 2026,Group Stage - Group A,Mexico,South Africa,2,0,Win,16,...,2,39,341.000,83.000,11,2,2,1,1,Full-time
1,WC2026_MEX_RSA_20260611,2026-06-11,FIFA World Cup 2026,Group Stage - Group A,South Africa,Mexico,0,2,Loss,3,...,5,61,538.000,91.000,12,1,1,1,3,Full-time
2,WC2026_KOR_CZE_20260612,2026-06-12,FIFA World Cup 2026,Group Stage - Group A,South Korea,Czechia,2,1,Win,15,...,4,38,314.000,70.000,16,0,0,2,5,Full-time
3,WC2026_KOR_CZE_20260612,2026-06-12,FIFA World Cup 2026,Group Stage - Group A,Czechia,South Korea,1,2,Loss,8,...,7,62,473.000,87.000,9,1,0,2,4,Full-time
4,WC2026_CAN_BIH_20260612,2026-06-12,FIFA World Cup 2026,Group Stage - Group B,Canada,Bosnia and Herzegovina,1,1,Draw,13,...,4,40,230.000,67.000,20,3,0,0,4,Full-time


In [7]:
worldcupstat.describe()

,team_goals,opponent_goals,shots,shots_on_target,possession_pct,passes,pass_accuracy_pct,fouls,yellow_cards,red_cards,...,opponent_shots,opponent_shots_on_target,opponent_possession_pct,opponent_passes,opponent_pass_accuracy_pct,opponent_fouls,opponent_yellow_cards,opponent_red_cards,opponent_offsides,opponent_corners
count,336.000,336.000,336.000,336.000,336.000,334.000,334.000,336.000,336.000,336.000,...,336.000,336.000,336.000,334.000,334.000,336.000,336.000,336.000,336.000,336.000
mean,1.429,1.429,11.812,4.366,50.000,446.916,86.677,11.729,1.464,0.057,...,11.812,4.366,50.000,446.916,86.677,11.729,1.464,0.057,1.851,4.533
std,1.374,1.374,6.037,2.854,12.781,144.501,5.955,4.302,1.347,0.256,...,6.037,2.854,12.781,144.501,5.955,4.302,1.347,0.256,1.625,2.984
min,0.000,0.000,0.000,0.000,18.000,137.000,57.000,1.000,0.000,0.000,...,0.000,0.000,18.000,137.000,57.000,1.000,0.000,0.000,0.000,0.000
25%,0.000,0.000,7.000,2.000,40.750,336.250,84.000,9.000,0.750,0.000,...,7.000,2.000,40.750,336.250,84.000,9.000,0.750,0.000,1.000,2.000
50%,1.000,1.000,11.000,4.000,50.000,439.000,88.000,11.000,1.000,0.000,...,11.000,4.000,50.000,439.000,88.000,11.000,1.000,0.000,1.500,4.000
75%,2.000,2.000,15.000,6.000,59.250,538.000,91.000,14.000,2.000,0.000,...,15.000,6.000,59.250,538.000,91.000,14.000,2.000,0.000,3.000,6.000
max,7.000,7.000,33.000,15.000,82.000,998.000,96.000,29.000,8.000,2.000,...,33.000,15.000,82.000,998.000,96.000,29.000,8.000,2.000,10.000,19.000


In [8]:
worldcupstat.columns

Index(['match_id', 'date', 'tournament', 'stage', 'team', 'opponent',
       'team_goals', 'opponent_goals', 'result', 'shots', 'shots_on_target',
       'possession_pct', 'passes', 'pass_accuracy_pct', 'fouls',
       'yellow_cards', 'red_cards', 'offsides', 'corners', 'opponent_shots',
       'opponent_shots_on_target', 'opponent_possession_pct',
       'opponent_passes', 'opponent_pass_accuracy_pct', 'opponent_fouls',
       'opponent_yellow_cards', 'opponent_red_cards', 'opponent_offsides',
       'opponent_corners', 'stats_period'],
      dtype='str')

In [9]:
match_id = worldcupstat["match_id"].iloc[0]
worldcupstat.loc[ worldcupstat["match_id"] == match_id,
    [
        "match_id",
        "team",
        "opponent",
        "team_goals",
        "opponent_goals",
        "shots",
        "possession_pct"
    ]
]

,match_id,team,opponent,team_goals,opponent_goals,shots,possession_pct
0,WC2026_MEX_RSA_20260611,Mexico,South Africa,2,0,16,61
1,WC2026_MEX_RSA_20260611,South Africa,Mexico,0,2,3,39


#### Merge rows for same match by creating seperate columsn for team1 and team2

In [10]:
def merge_matches(df):
    stats = ["team", "team_goals", "shots", "shots_on_target", "possession_pct", "passes", "pass_accuracy_pct", "fouls", "yellow_cards", "red_cards", "offsides", "corners"]

    df = df.copy()
    df["team_no"] = df.groupby("match_id").cumcount() + 1

    merged = df.pivot(index="match_id", columns="team_no", values=stats)

    merged.columns = [
        f"team{num}_{col}" if col != "team" else f"team{num}"
        for col, num in merged.columns
    ]

    return merged.reset_index()

In [11]:
merged_df = merge_matches(worldcupstat)

print(merged_df.shape)
display(merged_df.head())

(168, 25)


,match_id,team1,team2,team1_team_goals,team2_team_goals,team1_shots,team2_shots,team1_shots_on_target,team2_shots_on_target,team1_possession_pct,...,team1_fouls,team2_fouls,team1_yellow_cards,team2_yellow_cards,team1_red_cards,team2_red_cards,team1_offsides,team2_offsides,team1_corners,team2_corners
0,WC2022_ARG_AUS_20221203,Argentina,Australia,2,1,14,4,5,1,61,...,8,15,0,2,0,0,4,0,1,3
1,WC2022_ARG_CRO_20221213,Argentina,Croatia,3,0,9,11,7,2,39,...,15,7,2,2,0,0,1,0,2,4
2,WC2022_ARG_FRA_20221218,Argentina,France,3,3,21,8,10,4,53,...,23,16,5,3,0,0,5,4,6,5
3,WC2022_ARG_MEX_20221126,Argentina,Mexico,2,0,8,4,2,1,59,...,15,19,1,4,0,0,2,4,4,2
4,WC2022_ARG_SAU_20221122,Argentina,Saudi Arabia,1,2,13,3,7,2,68,...,7,18,0,6,0,0,10,1,9,2


#### Create target colums i.e. result where team 1 wins is 0, team 2 win is 1 and draw is 2

In [12]:
def create_target(df):
    df = df.copy()

    df["result"] = np.select(
        [
            df["team1_team_goals"] > df["team2_team_goals"],
            df["team1_team_goals"] < df["team2_team_goals"]
        ],
        [0, 1],
        default=2
    )

    return df

In [13]:
merged_df = create_target(merged_df)

display(merged_df[[
    "team1",
    "team2",
    "team1_team_goals",
    "team2_team_goals",
    "result"
]].head())

,team1,team2,team1_team_goals,team2_team_goals,result
0,Argentina,Australia,2,1,0
1,Argentina,Croatia,3,0,0
2,Argentina,France,3,3,2
3,Argentina,Mexico,2,0,0
4,Argentina,Saudi Arabia,1,2,1


In [14]:
print(merged_df["result"].value_counts().sort_index())

result
0    79
1    50
2    39
Name: count, dtype: int64


In [15]:
for column in merged_df.columns:
    print(column)

match_id
team1
team2
team1_team_goals
team2_team_goals
team1_shots
team2_shots
team1_shots_on_target
team2_shots_on_target
team1_possession_pct
team2_possession_pct
team1_passes
team2_passes
team1_pass_accuracy_pct
team2_pass_accuracy_pct
team1_fouls
team2_fouls
team1_yellow_cards
team2_yellow_cards
team1_red_cards
team2_red_cards
team1_offsides
team2_offsides
team1_corners
team2_corners
result


#### Check if there is any null

In [16]:
merged_df[merged_df.isnull().any(axis=1)]

,match_id,team1,team2,team1_team_goals,team2_team_goals,team1_shots,team2_shots,team1_shots_on_target,team2_shots_on_target,team1_possession_pct,...,team2_fouls,team1_yellow_cards,team2_yellow_cards,team1_red_cards,team2_red_cards,team1_offsides,team2_offsides,team1_corners,team2_corners,result
149,WC2026_SCO_BRA_20260624,Scotland,Brazil,0,3,11,21,5,8,46,...,10,1,2,0,0,0,1,7,6,1


In [17]:
print(merged_df[
    [
        "team1_passes",
        "team2_passes",
        "team1_pass_accuracy_pct",
        "team2_pass_accuracy_pct"
    ]
].dtypes)

team1_passes               object
team2_passes               object
team1_pass_accuracy_pct    object
team2_pass_accuracy_pct    object
dtype: object


#### Imput null data with median

In [18]:
X_cleaned = merged_df.copy()

In [19]:
exclude = ["match_id", "team1", "team2"]

In [20]:
numeric_cols = X_cleaned.columns.difference(exclude)

In [21]:
X_cleaned[numeric_cols] = X_cleaned[numeric_cols].apply(pd.to_numeric, errors="coerce")

In [22]:
X_cleaned[numeric_cols] = X_cleaned[numeric_cols].fillna(X_cleaned[numeric_cols].median())

In [23]:
print(X_cleaned.isna().sum()[X_cleaned.isna().sum() > 0])
display(X_cleaned.loc[[149]])

Series([], dtype: int64)


,match_id,team1,team2,team1_team_goals,team2_team_goals,team1_shots,team2_shots,team1_shots_on_target,team2_shots_on_target,team1_possession_pct,...,team2_fouls,team1_yellow_cards,team2_yellow_cards,team1_red_cards,team2_red_cards,team1_offsides,team2_offsides,team1_corners,team2_corners,result
149,WC2026_SCO_BRA_20260624,Scotland,Brazil,0,3,11,21,5,8,46,...,10,1,2,0,0,0,1,7,6,1


In [24]:
X_cleaned.to_csv("dataset/worldcupstat_merged.csv", index=False)